In [1]:
import os
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
import numpy as np
from sklearn.metrics import f1_score


In [2]:
def prepare_data_generators(data_dir, image_size=(128, 128), batch_size=32):
    train_datagen = ImageDataGenerator(rescale=1./255, validation_split=0.05)
    train_generator = train_datagen.flow_from_directory(
        data_dir, target_size=image_size, batch_size=batch_size, class_mode='categorical', subset='training'
    )
    val_generator = train_datagen.flow_from_directory(
        data_dir, target_size=image_size, batch_size=batch_size, class_mode='categorical', subset='validation'
    )
    class_labels = list(train_generator.class_indices.keys())  # Save class labels
    return train_generator, val_generator, class_labels

In [3]:
def build_mobilenetv2_model(input_shape=(128, 128, 3), num_classes=10):
    base_model = keras.applications.MobileNetV2(input_shape=input_shape, include_top=False, weights='imagenet')
    base_model.trainable = False  # Freeze base model layers
    
    model = keras.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ])
    
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

In [4]:
def train_model(model, train_generator, val_generator, epochs=50):
    model.fit(train_generator, validation_data=val_generator, epochs=epochs)
    return model

In [5]:
def evaluate_f1(model, val_generator):
    y_true = []
    y_pred = []
    
    for images, labels in val_generator:
        predictions = model.predict(images)
        y_true.extend(np.argmax(labels, axis=1))
        y_pred.extend(np.argmax(predictions, axis=1))
        
        if len(y_true) >= val_generator.samples:
            break
    
    f1 = f1_score(y_true, y_pred, average='weighted')
    print(f"Validation F1 Score: {f1:.4f}")
    return f1

In [6]:
def convert_to_tflite(model, tflite_path="models/yoga/mobilenetv2_yoga_quantized.tflite", class_labels_path="models/yoga/class_labels.npy"):
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    tflite_model = converter.convert()
    with open(tflite_path, "wb") as f:
        f.write(tflite_model)
    np.save(class_labels_path, class_labels)
    print(f"Quantized model saved at {tflite_path}")
    print(f"Class labels saved at {class_labels_path}")

In [7]:
def load_tflite_model(tflite_path="models/yoga/mobilenetv2_yoga_quantized.tflite", class_labels_path="models/yoga/class_labels.npy"):
    interpreter = tf.lite.Interpreter(model_path=tflite_path)
    interpreter.allocate_tensors()
    class_labels = np.load(class_labels_path, allow_pickle=True)
    return interpreter, class_labels.tolist()

In [8]:
def predict_image_tflite(interpreter, image_path, class_labels, image_size=(128, 128)):
    img = load_img(image_path, target_size=image_size)
    img_array = img_to_array(img) / 255.0
    img_array = np.expand_dims(img_array, axis=0).astype(np.float32)  # Add batch dimensionk
    
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()
    
    interpreter.set_tensor(input_details[0]['index'], img_array)
    interpreter.invoke()
    predictions = interpreter.get_tensor(output_details[0]['index'])
    
    predicted_class = np.argmax(predictions, axis=1)[0]
    return class_labels[predicted_class]

In [9]:
data_dir = "data/yoga/"  # Change this to your dataset path
train_generator, val_generator, class_labels = prepare_data_generators(data_dir)
model = build_mobilenetv2_model(num_classes=len(class_labels))
model = train_model(model, train_generator, val_generator)
f1 = evaluate_f1(model, val_generator)
convert_to_tflite(model)
    
    # Load TFLite model and test prediction


Found 5747 images belonging to 107 classes.
Found 244 images belonging to 107 classes.


D:\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/50
 41/180 ━━━━━━━━━━━━━━━━━━━━ 36s 262ms/step - accuracy: 0.0242 - loss: 4.9536

D:\Lib\site-packages\PIL\Image.py:1056: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


180/180 ━━━━━━━━━━━━━━━━━━━━ 65s 309ms/step - accuracy: 0.0270 - loss: 4.7206 - val_accuracy: 0.1557 - val_loss: 3.9157
Epoch 2/50
180/180 ━━━━━━━━━━━━━━━━━━━━ 53s 293ms/step - accuracy: 0.1173 - loss: 3.8586 - val_accuracy: 0.3607 - val_loss: 2.8654
Epoch 3/50
180/180 ━━━━━━━━━━━━━━━━━━━━ 53s 294ms/step - accuracy: 0.2405 - loss: 3.0593 - val_accuracy: 0.4139 - val_loss: 2.3801
Epoch 4/50
180/180 ━━━━━━━━━━━━━━━━━━━━ 53s 297ms/step - accuracy: 0.3202 - loss: 2.6080 - val_accuracy: 0.5082 - val_loss: 2.0979
Epoch 5/50
180/180 ━━━━━━━━━━━━━━━━━━━━ 51s 283ms/step - accuracy: 0.3736 - loss: 2.3104 - val_accuracy: 0.5205 - val_loss: 1.9869
Epoch 6/50
180/180 ━━━━━━━━━━━━━━━━━━━━ 48s 267ms/step - accuracy: 0.4316 - loss: 2.0699 - val_accuracy: 0.5451 - val_loss: 1.8656
Epoch 7/50
180/180 ━━━━━━━━━━━━━━━━━━━━ 46s 254ms/step - accuracy: 0.4458 - loss: 1.9359 - val_accuracy: 0.5287 - val_loss: 1.8151
Epoch 8/50
180/180 ━━━━━━━━━━━━━━━━━━━━ 47s 263ms/step - accuracy: 0.4917 - loss: 1.7923 - val

INFO:tensorflow:Assets written to: C:\Users\HP\AppData\Local\Temp\tmpb2ttx1cv\assets


Saved artifact at 'C:\Users\HP\AppData\Local\Temp\tmpb2ttx1cv'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 128, 128, 3), dtype=tf.float32, name='keras_tensor_154')
Output Type:
  TensorSpec(shape=(None, 107), dtype=tf.float32, name=None)
Captures:
  1978155588624: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1978155589392: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1978155589776: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1978155589008: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1978155587664: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1978155589584: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1978155589200: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1978155587856: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1978155590352: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1978155588048: TensorSpec(shape=(), dtype=tf.resource, name=None)
  19

In [ ]:

interpreter, class_labels = load_tflite_model()
test_image_path = "data/excercies_images/pull up/pullup_100061.jpg"  # Change this to an actual image path
predicted_label = predict_image_tflite(interpreter=interpreter,image_path=test_image_path, class_labels=class_labels)
print(f"Predicted class: {predicted_label}")